In [3]:
import torch
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
from tqdm import tqdm
import os
import shutil  # Thêm thư viện này để copy file

# Load VAD model
print("Loading model VAD...")
vad_model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                    model='silero_vad',
                                    force_reload=False,
                                    trust_repo=True)
get_speech_timestamps, _, _, _, _ = utils

def rms_normalize(waveform, target_rms=0.05, eps=1e-8):
    rms = torch.sqrt(torch.mean(waveform ** 2) + eps)
    gain = target_rms / rms
    return waveform * gain

def preprocess_audio_folder(
    input_dir,
    output_dir,
    target_sample_rate=16000,
    vad_model=None,
    get_speech_timestamps=None):
    """
    Preprocess all wav files in a folder and save results.
    If unprocessed/skipped, copy the original file to maintain the file count.
    """

    assert vad_model is not None, "vad_model is required"
    assert get_speech_timestamps is not None, "get_speech_timestamps function is required"

    input_dir = os.path.abspath(input_dir)
    os.makedirs(output_dir, exist_ok=True)

    resampler_cache = {}

    for root, _, files in os.walk(input_dir):
        for file in tqdm(files, desc="Processing audio"):
            in_path = os.path.join(root, file)
            rel_path = os.path.relpath(root, input_dir)
            out_dir = os.path.join(output_dir, rel_path)
            os.makedirs(out_dir, exist_ok=True)
            out_path = os.path.join(out_dir, file)

            # Nếu không phải file .wav, copy thẳng qua luôn để không rớt file (ví dụ .txt, .json)
            if not file.lower().endswith(".wav"):
                shutil.copy2(in_path, out_path)
                continue

            try:
                # Load audio
                waveform, orig_sr = torchaudio.load(in_path)

                # Trường hợp 1: File rỗng -> copy file gốc
                if waveform.numel() == 0:
                    shutil.copy2(in_path, out_path)
                    continue

                # Resample
                if orig_sr != target_sample_rate:
                    if orig_sr not in resampler_cache:
                        resampler_cache[orig_sr] = T.Resample(orig_sr, target_sample_rate)
                    waveform = resampler_cache[orig_sr](waveform)

                # Mono
                if waveform.shape[0] > 1:
                    waveform = torch.mean(waveform, dim=0, keepdim=True)

                # High-pass filter (>80Hz)
                waveform = F.highpass_biquad(
                    waveform,
                    sample_rate=target_sample_rate,
                    cutoff_freq=80)

                # RMS Normalize
                waveform = rms_normalize(waveform)

                # VAD
                wav_1d = waveform.squeeze()
                speech_timestamps = get_speech_timestamps(
                    wav_1d,
                    vad_model,
                    sampling_rate=target_sample_rate)

                # Trường hợp 2: Không phát hiện giọng nói -> copy file gốc
                if len(speech_timestamps) == 0:
                    shutil.copy2(in_path, out_path)
                    continue

                speech_segments = [
                    wav_1d[ts["start"]:ts["end"]]
                    for ts in speech_timestamps
                    if ts["end"] > ts["start"]
                ]

                # Trường hợp 3: Không có segment hợp lệ -> copy file gốc
                if len(speech_segments) == 0:
                    shutil.copy2(in_path, out_path)
                    continue

                # Nếu mọi thứ đều OK, gộp các segment có tiếng lại và lưu
                clean_waveform = torch.cat(speech_segments).unsqueeze(0)
                clean_waveform = clean_waveform.clamp(-1.0, 1.0)

                torchaudio.save(out_path, clean_waveform, target_sample_rate)

            except Exception as e:
                # Trường hợp 4: File bị lỗi đọc/ghi -> vẫn copy file gốc qua
                print(f"\n⚠️ Lỗi ở file {in_path}: {e} -> Đang copy nguyên bản.")
                shutil.copy2(in_path, out_path)

    print(f"✅ Hoàn tất! Dữ liệu đã lưu tại → {output_dir}")

Loading model VAD...


Using cache found in C:\Users\Lenovo/.cache\torch\hub\snakers4_silero-vad_master


### Preprocessing for VSASV

In [4]:
preprocess_audio_folder(
    input_dir=r"D:\Study\7-SP26\DATxSLP\Test set O\test-O",
    output_dir=r"D:\Study\7-SP26\DATxSLP\Test set O\test_o_clean",
    target_sample_rate=16000,
    vad_model=vad_model,
    get_speech_timestamps=get_speech_timestamps)

Processing audio:   0%|          | 0/17786 [00:00<?, ?it/s]

Processing audio: 100%|██████████| 17786/17786 [28:58<00:00, 10.23it/s] 

✅ Hoàn tất! Dữ liệu đã lưu tại → D:\Study\7-SP26\DATxSLP\Test set O\test_o_clean


### Preprocessing for VoxVietnam

In [ ]:
preprocess_audio_folder(
    input_dir=r"speech_data\wav\Vox_train",
    output_dir=r"speech_data\clean_wav\Vox_train",
    target_sample_rate=16000,
    vad_model=vad_model,
    get_speech_timestamps=get_speech_timestamps)